In [ ]:
import os, shutil, zipfile
from roboflow import Roboflow

In [2]:

# Download and unzip Dataset1.1
rf = Roboflow(api_key="jAz5EeLfcszRUC74qLkS")
ds = rf.workspace("surfline").project("surfer-spotting").version(2).download("yolov5")
shutil.move(ds.location, "Dataset1.1")

zip_path = "Dataset1.1/roboflow.zip"
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("Dataset1.1")

print("Dataset1.1 downloaded and extracted successfully.")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Surfer-Spotting-2 in yolov5pytorch:: 100%|██████████| 71100/71100 [00:24<00:00, 2866.46it/s]


Dataset1.1 downloaded and extracted successfully.


In [3]:

# Create combined data directory for Dataset1.1
combined_path = "CombinedData_Dataset1.1"
os.makedirs(f"{combined_path}/train/images", exist_ok=True)
os.makedirs(f"{combined_path}/train/labels", exist_ok=True)

# Copy training data from Dataset1.1
def copy_data(from_dir):
    shutil.copytree(f"{from_dir}/train/images", f"{combined_path}/train/images", dirs_exist_ok=True)
    shutil.copytree(f"{from_dir}/train/labels", f"{combined_path}/train/labels", dirs_exist_ok=True)

copy_data("Dataset1.1")

# Clean labels (keep only class 0)
def clean_labels(folder):
    for fname in os.listdir(folder):
        if fname.endswith(".txt"):
            path = os.path.join(folder, fname)
            with open(path, "r") as f:
                lines = f.readlines()
            kept = [line for line in lines if line.strip().split()[0] == "0"]
            if kept:
                with open(path, "w") as f:
                    f.writelines(kept)
            else:
                os.remove(path)

clean_labels(f"{combined_path}/train/labels")
print("Dataset1.1 merged and labels cleaned.")


Dataset1.1 merged and labels cleaned.


In [4]:

# Create data.yaml with absolute path
abs_path = os.path.abspath(f"{combined_path}/train/images")
yaml_text = f"""
train: {abs_path}
val: {abs_path}
nc: 1
names: ['surfer']
"""

with open(f"{combined_path}/data.yaml", "w") as f:
    f.write(yaml_text.strip())

print("Absolute paths written to data.yaml.")


Absolute paths written to data.yaml.


In [ ]:

# Start training
!python yolov5/train.py --img 640 --batch 8 --epochs 20 --data CombinedData_Dataset1.1/data.yaml --weights yolov5s.pt --name surfer-model-dataset1.1 --project CombinedData_Dataset1.1